# 04 — Embeddings & Sentence Transformer Fine-tuning

**Phase 4 deliverable (part 1).**

Fine-tunes `neuralmind/bert-base-portuguese-cased` as a sentence transformer
using contrastive pairs from CVM filing chunks.

## Goals
- Generate contrastive training pairs from parsed chunks
- Fine-tune with `MultipleNegativesRankingLoss` on RTX A1000 (fp16)
- Evaluate retrieval improvement over base model on the test set
- Upload fine-tuned model to HuggingFace Hub

In [1]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

from src import config
from src.nlp.train_sentence_transformer import generate_training_pairs

# Load cached pairs (generated by train_sentence_transformer.py --pairs-cache)
pairs_path = config.PROCESSED_DIR / "training_pairs.json"
if pairs_path.exists():
    raw = json.loads(pairs_path.read_text())
    pairs = [tuple(p) for p in raw]
    print(f"Loaded {len(pairs):,} training pairs from cache")
else:
    print("Generating pairs (this may take ~30s)…")
    pairs = generate_training_pairs(config.CHUNKS_DIR)
    print(f"Generated {len(pairs):,} pairs")

anchors = [a for a, _ in pairs]
positives = [p for _, p in pairs]

anchor_lengths = [len(t.split()) for t in anchors]
positive_lengths = [len(t.split()) for t in positives]

print(f"\nAnchor token counts:   min={min(anchor_lengths)}  mean={np.mean(anchor_lengths):.0f}  max={max(anchor_lengths)}")
print(f"Positive token counts: min={min(positive_lengths)}  mean={np.mean(positive_lengths):.0f}  max={max(positive_lengths)}")


Loaded 123,967 training pairs from cache



Anchor token counts:   min=38  mean=260  max=1288
Positive token counts: min=38  mean=259  max=1288


## 1. Training Pair Statistics

In [2]:
fig = go.Figure()
fig.add_trace(go.Histogram(x=anchor_lengths, name="Anchor", opacity=0.7, nbinsx=50))
fig.add_trace(go.Histogram(x=positive_lengths, name="Positive", opacity=0.7, nbinsx=50))
fig.update_layout(
    barmode="overlay",
    title="Token Count Distribution — Training Pairs",
    xaxis_title="Words per chunk",
    yaxis_title="Count",
    xaxis=dict(range=[0, 500]),
)
fig.show()


## 2. Fine-Tuning — Training Loss Curve

Run the trainer from the command line (takes ~2–4 h on RTX A1000):

```bash
python -m src.nlp.train_sentence_transformer --epochs 10 --batch-size 16
```

The trainer writes checkpoints and a `trainer_state.json` to `models/sentence_transformer/`.

In [3]:
model_dir = config.MODELS_DIR / "sentence_transformer"
trainer_state_path = model_dir / "trainer_state.json"

if trainer_state_path.exists():
    state = json.loads(trainer_state_path.read_text())
    log_history = state.get("log_history", [])

    train_logs = [l for l in log_history if "loss" in l and "eval_loss" not in l]
    eval_logs  = [l for l in log_history if "eval_loss" in l]

    fig = go.Figure()
    if train_logs:
        fig.add_trace(go.Scatter(
            x=[l["step"] for l in train_logs],
            y=[l["loss"] for l in train_logs],
            mode="lines", name="Train Loss",
        ))
    fig.update_layout(title="Training Loss vs Steps", xaxis_title="Step", yaxis_title="Loss")
    fig.show()

    if eval_logs:
        fig2 = go.Figure()
        fig2.add_trace(go.Scatter(
            x=[l["step"] for l in eval_logs],
            y=[l["eval_loss"] for l in eval_logs],
            mode="lines+markers", name="Eval Loss",
            line=dict(color="green"),
        ))
        fig2.update_layout(title="Eval Loss vs Steps (MNR, lower = better)", xaxis_title="Step", yaxis_title="Eval Loss")
        fig2.show()

    best_metric = state.get("best_metric")
    best_step = state.get("best_model_checkpoint", "")
    print(f"Best eval loss: {best_metric:.4f}" if best_metric else "Training in progress…")
    print(f"Best checkpoint: {best_step}")
else:
    print("Training not yet complete — run the trainer first.")
    print(f"Expected path: {trainer_state_path}")


Training not yet complete — run the trainer first.
Expected path: /home/conderafael/Programing/Portfolio/cvm-intelligence/models/sentence_transformer/trainer_state.json


## 3. Embedding Quality — Cosine Similarity on Sample Pairs

Compare cosine similarity for positive pairs (should be high) vs random pairs (should be low)
using both the base model and the fine-tuned model.

In [4]:
from src.nlp.embedder import load_model
import random

# Sample 200 positive pairs and 200 random (negative) pairs
rng = random.Random(0)
sample_size = 200
sample_pos = rng.sample(pairs, sample_size)
all_texts = anchors + positives
random_positives = [rng.choice(all_texts) for _ in range(sample_size)]
sample_neg = [(a, r) for (a, _), r in zip(sample_pos, random_positives)]

def cosine_sim_batch(model, pairs_list):
    texts_a = [a for a, _ in pairs_list]
    texts_b = [b for _, b in pairs_list]
    emb_a = model.encode(texts_a, batch_size=32, show_progress_bar=False, normalize_embeddings=True)
    emb_b = model.encode(texts_b, batch_size=32, show_progress_bar=False, normalize_embeddings=True)
    return (emb_a * emb_b).sum(axis=1)

print("Loading base model…")
base_model = load_model(model_dir=None)
base_pos_sims  = cosine_sim_batch(base_model, sample_pos)
base_neg_sims  = cosine_sim_batch(base_model, sample_neg)

fine_tuned_pos_sims = fine_tuned_neg_sims = None
if model_dir.exists():
    print("Loading fine-tuned model…")
    fine_model = load_model(model_dir=model_dir)
    fine_tuned_pos_sims = cosine_sim_batch(fine_model, sample_pos)
    fine_tuned_neg_sims = cosine_sim_batch(fine_model, sample_neg)

print(f"\nBase model:        positive mean={base_pos_sims.mean():.3f}  negative mean={base_neg_sims.mean():.3f}  gap={base_pos_sims.mean()-base_neg_sims.mean():.3f}")
if fine_tuned_pos_sims is not None:
    print(f"Fine-tuned model:  positive mean={fine_tuned_pos_sims.mean():.3f}  negative mean={fine_tuned_neg_sims.mean():.3f}  gap={fine_tuned_pos_sims.mean()-fine_tuned_neg_sims.mean():.3f}")


Loading base model…


/home/conderafael/Programing/Portfolio/cvm-intelligence/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 57333.87it/s]


BertModel LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading fine-tuned model…


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 15658.61it/s]


Base model:        positive mean=0.892  negative mean=0.830  gap=0.061
Fine-tuned model:  positive mean=0.517  negative mean=0.073  gap=0.444


In [5]:
rows = []
for label, sims in [("Base — Positive", base_pos_sims), ("Base — Random (neg)", base_neg_sims)]:
    for s in sims:
        rows.append({"model": "Base BERTimbau", "pair_type": label.split(" — ")[1], "cosine_sim": float(s)})

if fine_tuned_pos_sims is not None:
    for label, sims in [("Fine-tuned — Positive", fine_tuned_pos_sims), ("Fine-tuned — Random (neg)", fine_tuned_neg_sims)]:
        for s in sims:
            rows.append({"model": "Fine-tuned BERTimbau", "pair_type": label.split(" — ")[1], "cosine_sim": float(s)})

df_sim = pd.DataFrame(rows)

fig = px.box(
    df_sim,
    x="model", y="cosine_sim", color="pair_type",
    title="Cosine Similarity: Positive vs Random Pairs",
    labels={"cosine_sim": "Cosine Similarity", "model": "Model", "pair_type": "Pair Type"},
    points="outliers",
)
fig.show()


## 4. Model Summary

In [6]:
print("=== Training Configuration ===")
print(f"  Base model:           neuralmind/bert-base-portuguese-cased")
print(f"  Max sequence length:  {config.MAX_SEQ_LENGTH} tokens")
print(f"  Batch size:           {config.FINETUNE_BATCH_SIZE}")
print(f"  Gradient accumulation:{config.GRADIENT_ACCUMULATION_STEPS}  (effective batch = {config.FINETUNE_BATCH_SIZE * config.GRADIENT_ACCUMULATION_STEPS})")
print(f"  Epochs:               10")
print(f"  Learning rate:        2e-5")
print(f"  FP16:                 True")
print(f"  Loss:                 MultipleNegativesRankingLoss (in-batch negatives)")
print()
print("=== Training Data ===")
print(f"  Total pairs:          {len(pairs):,}")
print(f"  Train pairs:          {len(pairs) - 500:,}")
print(f"  Eval pairs:           500")
print()
print("=== Output ===")
print(f"  Model saved to:       {model_dir}")
print(f"  Model exists:         {model_dir.exists()}")
if (model_dir / "config.json").exists():
    cfg = json.loads((model_dir / "config.json").read_text())
    print(f"  Hidden size:          {cfg.get('hidden_size', 'N/A')}")
    print(f"  Num parameters:       ~110M (BERTimbau base)")


=== Training Configuration ===
  Base model:           neuralmind/bert-base-portuguese-cased
  Max sequence length:  256 tokens
  Batch size:           16
  Gradient accumulation:4  (effective batch = 64)
  Epochs:               10
  Learning rate:        2e-5
  FP16:                 True
  Loss:                 MultipleNegativesRankingLoss (in-batch negatives)

=== Training Data ===
  Total pairs:          123,967
  Train pairs:          123,467
  Eval pairs:           500

=== Output ===
  Model saved to:       /home/conderafael/Programing/Portfolio/cvm-intelligence/models/sentence_transformer
  Model exists:         True
  Hidden size:          768
  Num parameters:       ~110M (BERTimbau base)


## Embedding Comparison — Fine-tuned vs Baselines

Evaluates three sentence transformer models against the retrieval test set
(94 queries, labeled relevant chunk IDs) using **dense-only** retrieval to isolate
embedding quality from BM25 and reranking effects.

**Fine-tuned model** is evaluated on both:
1. The full 97,138-chunk ChromaDB corpus (no re-embedding — uses pre-built index)
2. The same 5,000-chunk subsample as the baselines (cross-check)

**Base models** use a 5,000-chunk subsample:
- All 274 relevant chunks are guaranteed present
- Padded with random SQLite chunks to reach 5,000
- In-memory cosine search via numpy (no FAISS needed at this scale)

Comparing all three on the **same 5,000-chunk pool** gives a strict apples-to-apples result.


In [ ]:
with open(config.RETRIEVAL_TEST_SET_PATH) as f:
    data = json.load(f)
queries = data["queries"]
print(f"Test set: {len(queries)} queries")
print(f"Query types: {set(q['query_type'] for q in queries)}")


In [ ]:
import sqlite3

SUBSAMPLE_SIZE = 5_000

# Collect all chunk IDs that appear in any relevant_chunk_ids list
required_ids = set()
for q in queries:
    required_ids.update(q["relevant_chunk_ids"])
print(f"Required (relevant) chunks: {len(required_ids)}")

# Fetch required chunk texts from SQLite
n_extra = SUBSAMPLE_SIZE - len(required_ids)
conn = sqlite3.connect(config.DB_PATH)

req_rows = conn.execute(
    f"SELECT chunk_id, chunk_text FROM chunks "
    f"WHERE chunk_id IN ({','.join('?' * len(required_ids))})",
    tuple(required_ids),
).fetchall()

extra_rows = conn.execute(
    f"SELECT chunk_id, chunk_text FROM chunks "
    f"WHERE chunk_text IS NOT NULL AND chunk_text != '' "
    f"AND chunk_id NOT IN ({','.join('?' * len(required_ids))}) "
    f"ORDER BY RANDOM() LIMIT ?",
    (*required_ids, n_extra),
).fetchall()
conn.close()

subsample_ids   = [r[0] for r in req_rows] + [r[0] for r in extra_rows]
subsample_texts = [r[1] for r in req_rows] + [r[1] for r in extra_rows]
print(f"Subsample built: {len(subsample_ids)} chunks ({len(req_rows)} required + {len(extra_rows)} random)")
print("All relevant chunks present:", all(cid in subsample_ids for cid in required_ids))


In [ ]:
import math
import torch
from tqdm.auto import tqdm

# ── IR metric functions ───────────────────────────────────────────────────────

def recall_at_k(retrieved_ids: list, relevant_ids: set, k: int) -> float:
    """Proportion of relevant docs found in the top-k retrieved."""
    if not relevant_ids:
        return 0.0
    return len(set(retrieved_ids[:k]) & relevant_ids) / len(relevant_ids)


def reciprocal_rank(retrieved_ids: list, relevant_ids: set) -> float:
    """Reciprocal of the rank of the first relevant document."""
    for rank, doc_id in enumerate(retrieved_ids, start=1):
        if doc_id in relevant_ids:
            return 1.0 / rank
    return 0.0


def ndcg_at_k(retrieved_ids: list, relevant_ids: set, k: int) -> float:
    """Normalised Discounted Cumulative Gain at k."""
    dcg = sum(
        1.0 / math.log2(r + 2)
        for r, doc_id in enumerate(retrieved_ids[:k])
        if doc_id in relevant_ids
    )
    ideal = sum(1.0 / math.log2(r + 2) for r in range(min(len(relevant_ids), k)))
    return dcg / ideal if ideal else 0.0


def evaluate_in_memory(
    queries: list,
    chunk_ids: list,
    chunk_embs: np.ndarray,
    query_embs: np.ndarray,
) -> dict:
    """Dense cosine retrieval over an in-memory embedding matrix."""
    chunk_ids_arr = np.array(chunk_ids)
    r5, r10, mrr_vals, ndcg_vals = [], [], [], []
    for i, q in enumerate(queries):
        relevant = set(q["relevant_chunk_ids"])
        scores = chunk_embs @ query_embs[i]         # (N,) — L2-normalised dot = cosine
        top_idx = np.argsort(scores)[::-1][:10]
        top_ids = chunk_ids_arr[top_idx].tolist()
        r5.append(recall_at_k(top_ids, relevant, 5))
        r10.append(recall_at_k(top_ids, relevant, 10))
        mrr_vals.append(reciprocal_rank(top_ids, relevant))
        ndcg_vals.append(ndcg_at_k(top_ids, relevant, 10))
    return {
        "Recall@5":  float(np.mean(r5)),
        "Recall@10": float(np.mean(r10)),
        "MRR":       float(np.mean(mrr_vals)),
        "NDCG@10":   float(np.mean(ndcg_vals)),
    }


print("Metric functions defined.")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}  |  VRAM: {torch.cuda.get_device_properties(0).total_memory // 1024**3} GB")


In [ ]:
# ── Fine-tuned BERTimbau ──────────────────────────────────────────────────────
# Evaluated on: (1) the same 5k subsample and (2) full ChromaDB corpus (97k)
from sentence_transformers import SentenceTransformer
from src.rag.indexer import load_chroma_collection
from src.rag.retriever import retrieve_dense

collection = load_chroma_collection(config.CHROMADB_DIR, config.CHROMA_COLLECTION_NAME)

print("Loading fine-tuned model…")
ft_model = SentenceTransformer(str(config.MODELS_DIR / "sentence_transformer"), device="cuda")
ft_model.max_seq_length = config.MAX_SEQ_LENGTH

print("Embedding subsample with fine-tuned model…")
ft_sub_embs = ft_model.encode(
    subsample_texts, batch_size=64, show_progress_bar=True,
    normalize_embeddings=True, convert_to_numpy=True,
).astype(np.float32)

print("Embedding queries with fine-tuned model…")
ft_query_embs = ft_model.encode(
    [q["query"] for q in queries], batch_size=64, show_progress_bar=True,
    normalize_embeddings=True, convert_to_numpy=True,
).astype(np.float32)

ft_subsample_metrics = evaluate_in_memory(queries, subsample_ids, ft_sub_embs, ft_query_embs)
print("Fine-tuned (5k subsample):", ft_subsample_metrics)

# Full ChromaDB — reuse the already-computed query embeddings (no re-encoding)
print("\nEvaluating fine-tuned on full ChromaDB corpus…")
r5, r10, mrr_vals, ndcg_vals = [], [], [], []
for i, q in enumerate(tqdm(queries, desc="ChromaDB")):
    relevant = set(q["relevant_chunk_ids"])
    chunks = retrieve_dense(ft_query_embs[i], collection, top_k=10)
    top_ids = [c.chunk_id for c in chunks]
    r5.append(recall_at_k(top_ids, relevant, 5))
    r10.append(recall_at_k(top_ids, relevant, 10))
    mrr_vals.append(reciprocal_rank(top_ids, relevant))
    ndcg_vals.append(ndcg_at_k(top_ids, relevant, 10))
ft_full_metrics = {
    "Recall@5": float(np.mean(r5)), "Recall@10": float(np.mean(r10)),
    "MRR": float(np.mean(mrr_vals)), "NDCG@10": float(np.mean(ndcg_vals)),
}
print("Fine-tuned (97k full corpus):", ft_full_metrics)

del ft_model, ft_sub_embs, ft_query_embs
torch.cuda.empty_cache()
print("\nVRAM freed.")


In [ ]:
# ── Base BERTimbau (no fine-tuning) ──────────────────────────────────────────
print(f"Loading base model: {config.BERT_MODEL_NAME}…")
base_model = SentenceTransformer(config.BERT_MODEL_NAME, device="cuda")
base_model.max_seq_length = config.MAX_SEQ_LENGTH

print("Embedding subsample…")
base_sub_embs = base_model.encode(
    subsample_texts, batch_size=64, show_progress_bar=True,
    normalize_embeddings=True, convert_to_numpy=True,
).astype(np.float32)

print("Embedding queries…")
base_query_embs = base_model.encode(
    [q["query"] for q in queries], batch_size=64, show_progress_bar=True,
    normalize_embeddings=True, convert_to_numpy=True,
).astype(np.float32)

base_metrics = evaluate_in_memory(queries, subsample_ids, base_sub_embs, base_query_embs)
print("Base BERTimbau (5k subsample):", base_metrics)

del base_model, base_sub_embs, base_query_embs
torch.cuda.empty_cache()
print("VRAM freed.")


In [ ]:
# ── paraphrase-multilingual-MiniLM-L12-v2 ────────────────────────────────────
MULTILINGUAL_MODEL = "paraphrase-multilingual-MiniLM-L12-v2"
print(f"Loading {MULTILINGUAL_MODEL}…")
multi_model = SentenceTransformer(MULTILINGUAL_MODEL, device="cuda")
multi_model.max_seq_length = 256  # cap to match other models

print("Embedding subsample…")
multi_sub_embs = multi_model.encode(
    subsample_texts, batch_size=64, show_progress_bar=True,
    normalize_embeddings=True, convert_to_numpy=True,
).astype(np.float32)

print("Embedding queries…")
multi_query_embs = multi_model.encode(
    [q["query"] for q in queries], batch_size=64, show_progress_bar=True,
    normalize_embeddings=True, convert_to_numpy=True,
).astype(np.float32)

multi_metrics = evaluate_in_memory(queries, subsample_ids, multi_sub_embs, multi_query_embs)
print(f"{MULTILINGUAL_MODEL} (5k subsample):", multi_metrics)

del multi_model, multi_sub_embs, multi_query_embs
torch.cuda.empty_cache()
print("VRAM freed.")


In [ ]:
import datetime

# ── Compile and display results ───────────────────────────────────────────────
all_results = {
    "fine_tuned_full":      {"label": "Fine-tuned BERTimbau (97k corpus)",                     **ft_full_metrics},
    "fine_tuned_subsample": {"label": "Fine-tuned BERTimbau (5k subsample)",                   **ft_subsample_metrics},
    "base_bertimbau":       {"label": "Base BERTimbau (5k subsample)",                         **base_metrics},
    "multilingual_minilm":  {"label": "paraphrase-multilingual-MiniLM-L12-v2 (5k subsample)", **multi_metrics},
}

metric_keys = ["Recall@5", "Recall@10", "MRR", "NDCG@10"]
rows = [
    {"Model": v["label"], **{k: round(v[k], 4) for k in metric_keys}}
    for v in all_results.values()
]
results_df = pd.DataFrame(rows).set_index("Model")
display(results_df.style.highlight_max(axis=0, color="#d5f5e3"))

# Grouped bar chart — subsample configs only (strict apples-to-apples)
fig = go.Figure()
colors = ["#3498db", "#2ecc71", "#e67e22", "#9b59b6"]
sub_order = ["fine_tuned_subsample", "base_bertimbau", "multilingual_minilm"]
for metric, color in zip(metric_keys, colors):
    fig.add_trace(go.Bar(
        name=metric,
        x=[all_results[k]["label"].split(" (")[0] for k in sub_order],
        y=[all_results[k][metric] for k in sub_order],
        marker_color=color,
        text=[f"{all_results[k][metric]:.4f}" for k in sub_order],
        textposition="outside",
    ))
max_val = max(all_results[k]["Recall@10"] for k in sub_order)
fig.update_layout(
    barmode="group",
    title="Embedding quality — 5k subsample (apples-to-apples)",
    yaxis=dict(title="Score", range=[0, max_val * 1.45]),
    legend_title="Metric",
    height=440,
)
fig.show()

# Cross-check: fine-tuned subsample vs full corpus
print("\nFine-tuned full-corpus vs subsample cross-check:")
for k in metric_keys:
    delta = ft_full_metrics[k] - ft_subsample_metrics[k]
    print(f"  {k}: full={ft_full_metrics[k]:.4f}  sub={ft_subsample_metrics[k]:.4f}  Δ={delta:+.4f}")

# ── Save JSON ─────────────────────────────────────────────────────────────────
output = {
    "metadata": {
        "n_queries": len(queries),
        "subsample_size": len(subsample_ids),
        "n_relevant_chunks": len(required_ids),
        "subsample_seed": 42,
        "generated_at": datetime.datetime.now().isoformat(),
    },
    "results": all_results,
}
out_path = config.EVALUATION_DIR / "embedding_comparison.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)
print(f"\nSaved → {out_path}")
